# Protein Classification: Enzyme vs. Non-Enzyme Prediction using ProtBERT

## 1. Project Objective
The goal of this project is to build a deep learning classifier that can determine whether a given human protein functions as an enzyme, based solely on its raw amino acid sequence.

We will use **ProtBERT** to extract 1024-dimensional contextual embeddings from the sequences, and feed those numerical representations into a **Multi-Layer Perceptron (Neural Network)**.

## 2. Dataset Preparation
The raw dataset was downloaded from UniProt (Swiss-Prot). We define our classification target using the `EC number` (Enzyme Commission number) column:
* **Label 1 (Enzyme):** The protein has an assigned EC number.
* **Label 0 (Non-Enzyme):** The `EC number` column is empty.

To prevent class imbalance and keep computational times manageable, we randomly sample 1000 enzymes and 1000 non-enzymes to create a final dataset of 2000 proteins.

In [ ]:
import pandas as pd

# 1. Load the raw UniProt data
# The file is tab-separated, so we specify sep="\t"
df = pd.read_csv("../data/raw/uniprotkb_reviewed_true_AND_organism_id_2026_03_23.tsv.gz", sep="\t")

# 2. Create the binary label
# If the 'EC number' is not missing (notna), it's an enzyme (1). Otherwise, 0.
df["label"] = df["EC number"].notna().astype(int)

# 3. Sample exactly 1000 of each to create a balanced dataset
# random_state=42 ensures we get the exact same random sample every time we run this code
enzymes = df[df["label"] == 1].sample(1000, random_state=42)
non_enzymes = df[df["label"] == 0].sample(1000, random_state=42)

# Combine them into one dataset
dataset = pd.concat([enzymes, non_enzymes])

# 4. Keep only the required columns: Sequence and Label
dataset = dataset[["Sequence", "label"]]

# 5. Save the clean dataset to your directory
dataset.to_csv("enzyme_dataset.csv", index=False)

print(f"Dataset ready! Total proteins: {len(dataset)}")
print(dataset["label"].value_counts())

## 3. Feature Extraction using ProtBERT

In this step, we convert our raw amino acid sequences into dense numerical vectors (embeddings) using the `Rostlab/prot_bert` model. 

1. **Preprocessing:** ProtBERT expects sequences to be space-separated (e.g., "M K T I...").
2. **Tokenization:** Converts the string into token IDs.
3. **Embedding:** We pass the tokens through the pre-trained transformer. We extract the `last_hidden_state` and take the average (mean pooling) across the sequence length to generate a single, fixed-size 1024-dimensional vector for each protein.

In [ ]:
import torch
from transformers import BertTokenizer, BertModel
from tqdm import tqdm
import numpy as np

# 1. Load the pre-trained ProtBERT model and tokenizer from Hugging Face
model_name = "Rostlab/prot_bert"
print(f"Loading {model_name}...")

tokenizer = BertTokenizer.from_pretrained(model_name, do_lower_case=False)
model = BertModel.from_pretrained(model_name)
print("Model loaded successfully!")

# 2. Define the preprocessing function
# ProtBERT requires spaces between each amino acid
def preprocess(seq):
    return " ".join(list(seq))

# 3. Extract Embeddings
embeddings = []
print("Starting embedding extraction. This may take a while...")

# tqdm gives us a nice progress bar
for seq in tqdm(dataset["Sequence"]):
    
    # Add spaces
    processed_seq = preprocess(seq)
    
    # Tokenize and format for PyTorch
    inputs = tokenizer(processed_seq, return_tensors="pt", truncation=True, max_length=1024)
    
    # Disable gradient calculation for faster inference (we are just extracting, not training)
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Get the last hidden state, mean pool it, and convert to a numpy array
    embedding = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
    embeddings.append(embedding)

# 4. Convert lists to final Machine Learning arrays (X and y)
X = np.array(embeddings)
y = dataset["label"].values

print(f"Feature matrix X shape: {X.shape}") # Should be (2000, 1024)
print(f"Target vector y shape: {y.shape}")   # Should be (2000,)

In [ ]:
import numpy as np
import os

# Create the processed directory path if it doesn't exist
os.makedirs('../data/processed', exist_ok=True)

# Save the arrays
np.save('../data/processed/X_embeddings.npy', X)
np.save('../data/processed/y_labels.npy', y)

print("Embeddings and labels saved successfully!")